# 🇬🇭 Automating Data Ingestion Pipelines
## A Ghana Fintech Workshop

---

> *"Data is the new oil — and Ghana is sitting on a goldmine of mobile money transactions every single day."*

Welcome! This notebook is your **theory guide** for the workshop. It covers the concepts, vocabulary, and thinking you need before you write a single line of code. By the end, you'll understand:

- What a data ingestion pipeline is and why it matters
- The ETL pattern (Extract, Transform, Load) through a fintech lens
- Which AWS services power modern pipelines — and what each one does
- Best practices used by real fintech companies in Accra

**Estimated reading time:** ~1.5 hours  
**Prerequisites:** Basic Python knowledge, a curious mind  
**No AWS account required** for this notebook — we explain concepts only here.

---

## 📋 Table of Contents

1. [Welcome & Context](#section1) — Ghana's fintech boom and the data challenge
2. [What is a Data Ingestion Pipeline?](#section2) — Plain-language definitions and real examples
3. [ETL Deep Dive](#section3) — Extract, Transform, Load explained with MoMo data
4. [AWS Services for Pipelines](#section4) — S3, Lambda, DynamoDB, EventBridge, SNS, CloudWatch
5. [Best Practices & Compliance](#section5) — Bank of Ghana, Ghana Data Protection Act, idempotency

---

<a id='section1'></a>
# Section 1 — Welcome & Context
### *Why are we here, and why does this matter?*

---

## 🇬🇭 Ghana's Fintech Story

Ghana has one of Africa's most vibrant mobile money ecosystems. Consider these numbers:

| Metric | Figure |
|--------|--------|
| Mobile money accounts (2024) | **~19 million active accounts** |
| Daily MoMo transactions | **Millions per day** across all networks |
| MTN MoMo market share | **~65% of mobile money volume** |
| Regulatory body | **Bank of Ghana (BoG)** — Electronic Money Issuers (EMIs) framework |

Every one of those millions of daily transactions generates **data** — a record of who sent what, to whom, when, from where, and whether it succeeded.

> 💡 **Did you know?** MTN MoMo, Vodafone Cash, and AirtelTigo Money all operate under Bank of Ghana's Electronic Money Issuer (EMI) guidelines. Companies like **Zeepay**, **Hubtel**, and **ExpressPay** act as payment aggregators connecting merchants to these networks.

---

## 🔥 The Data Challenge

Here's the problem: all that transaction data is **scattered, messy, and arriving constantly**.

Think about what a typical Ghanaian fintech company deals with every day:

- **Agent transactions** coming in from hundreds of MoMo agents across regions (Kumasi, Tamale, Cape Coast, Accra…)
- **Merchant payments** from POS terminals in shops and restaurants
- **Interbank transfers** routed through GhIPSS
- **USSD-initiated transactions** that sometimes get cut off mid-session due to poor network coverage
- **Failed and reversed transactions** that need special accounting treatment
- **Reconciliation files** that need to balance with the Bank of Ghana's records by midnight

Doing all of this **manually** is a nightmare. That's why we automate.

---

## 🏗️ What We're Building Today

By the end of this workshop, you'll have built a complete, automated data ingestion pipeline:

```
┌─────────────────────────────────────────────────────────────────────┐
│               GHANA FINTECH DATA PIPELINE ARCHITECTURE              │
│                                                                     │
│  ┌──────────┐     ┌─────────────────┐     ┌──────────────────┐     │
│  │  DATA    │     │   S3 BUCKET     │     │   S3 BUCKET      │     │
│  │ SOURCES  │────▶│   (Raw Zone)    │────▶│ (Processed Zone) │     │
│  │          │     │                 │     │                  │     │
│  │ • MoMo   │     │ transactions/   │     │ transactions/    │     │
│  │   APIs   │     │ 2026/04/20/     │     │ 2026/04/20/      │     │
│  │ • Agent  │     │ batch_001.json  │     │ batch_001.parquet│     │
│  │   Files  │     └────────┬────────┘     └────────┬─────────┘     │
│  │ • USSD   │              │                        │               │
│  │   Logs   │              │ Triggers               │ Loads         │
│  └──────────┘              ▼                        ▼               │
│                   ┌────────────────┐      ┌─────────────────┐      │
│                   │  AWS LAMBDA    │      │   DYNAMODB      │      │
│                   │  (Transform)   │      │                 │      │
│                   │                │      │ momo_transactions│      │
│                   │ • Clean phones │      │                 │      │
│                   │ • Remove dupes │      │ Fast lookups by │      │
│                   │ • Std regions  │      │ txn_id or date  │      │
│                   │ • Flag failed  │      └─────────────────┘      │
│                   └───────┬────────┘                               │
│                           │                                        │
│              ┌────────────┴──────────────┐                         │
│              ▼                           ▼                         │
│   ┌──────────────────┐       ┌──────────────────────┐             │
│   │   CLOUDWATCH     │       │        SNS           │             │
│   │   (Monitoring)   │       │    (Alerts)          │             │
│   │                  │       │                      │             │
│   │ • Logs & Metrics │       │ • Email on failure   │             │
│   │ • Alarms         │       │ • SMS to team lead   │             │
│   └──────────────────┘       └──────────────────────┘             │
│                                                                     │
│                    ⏰ EVENTBRIDGE (Scheduler)                       │
│              Runs the whole thing every night at 10pm UTC           │
└─────────────────────────────────────────────────────────────────────┘
```

Don't worry if this looks complicated right now — by the end, every box will make perfect sense.

---

<a id='section2'></a>
# Section 2 — What is a Data Ingestion Pipeline?
### *Think of it like the trotro system — but for data*

---

## 🚌 The Trotro Analogy

You know how the trotro system works in Ghana?

- Passengers (data) need to get from **point A** (the source: a MoMo agent in Kumasi) to **point B** (the destination: a central database in Accra)
- There are **multiple routes** — some passengers take the Accra-Kumasi highway, some transfer at Nkawkaw
- The **mate** (your pipeline code) collects the fares (data), checks they're valid, and keeps a record
- The trotro **runs on a schedule** — not just whenever passengers feel like it
- Sometimes the engine breaks down (errors) — you need a backup plan

A **data ingestion pipeline** is exactly this: **an automated system that moves data from where it's created to where it's useful** — reliably, on schedule, with error handling.

---

## 📖 Formal Definition

> **Data Ingestion Pipeline**: An automated workflow that continuously or periodically collects raw data from one or more sources, processes it into a usable format, and delivers it to a storage or analytics system — with monitoring and error handling built in.

Key words:
- **Automated** — it runs without someone manually doing it
- **Continuously or periodically** — either all the time (streaming) or on a schedule (batch)
- **Processes** — it doesn't just move data, it cleans and transforms it
- **Monitoring and error handling** — it tells you when something goes wrong

---

## 🇬🇭 Real Ghana Fintech Examples

### Example 1 — Daily Agent Reconciliation
An MTN MoMo agent in Kumasi (Agent ID: AGT-KSI-042) processes 200 transactions a day. At 10pm, the system automatically:
1. Pulls all transactions for that agent from the day
2. Calculates total cash-in, cash-out, and agent commission
3. Matches against the agent's float balance
4. Flags any discrepancies
5. Sends a reconciliation report to the regional manager

### Example 2 — Merchant Payment Reporting
Hubtel processes payments for hundreds of merchants across Ghana — from Melcom stores in Accra to chop bars in Tamale. Every night, a pipeline:
1. Collects all merchant payment records from the day
2. Groups them by merchant and region
3. Calculates settlement amounts (transaction amount minus Hubtel's fee)
4. Loads into a database for the finance team to review
5. Triggers settlement instructions for the next morning

### Example 3 — GhIPSS Interbank Data
The Ghana Interbank Payment and Settlement System (GhIPSS) facilitates transfers between banks and mobile money wallets. Banks need to:
1. Receive GhIPSS settlement files (usually CSVs or ISO 8583 messages)
2. Transform them into the bank's internal format
3. Post entries to the correct accounts
4. Generate the end-of-day position report for Bank of Ghana

All three of these are data ingestion pipelines.

---

## 😩 The Manual Approach (The Antipattern)

Before pipelines, this is how data teams worked — and some still do:

```
1. Finance officer logs into the MoMo backend portal at 8am
2. Downloads yesterday's CSV manually
3. Opens it in Excel
4. Manually deletes duplicate rows ("I think this one is a duplicate...")
5. Creates a VLOOKUP to match region codes
6. Manually fixes phone number formats
7. Sends the Excel file to the manager via WhatsApp
8. Manager opens it, finds an error, sends it back
9. Repeat from step 4
10. By 11am, a decision that needed data from 8am has been delayed 3 hours
```

### Manual vs Automated — A Comparison

| Dimension | Manual (Excel + WhatsApp) | Automated Pipeline |
|-----------|--------------------------|--------------------|
| **Speed** | Hours to days | Minutes to seconds |
| **Reliability** | Human errors common | Consistent, tested logic |
| **Scale** | Falls apart at 10,000+ rows | Handles millions of records |
| **Audit trail** | "Which version did we use?" | Full logs, versioned data |
| **Availability** | Depends on who's in the office | Runs 24/7 |
| **Cost** | High (senior staff time) | Low (compute on-demand) |
| **Compliance** | Hard to prove what happened | Logs satisfy BoG requirements |

> ⚠️ **A real risk**: A finance manager in a Ghanaian fintech told us: *"We once paid out settlements based on a duplicate transaction row in Excel that nobody caught. It cost us GHS 47,000 to recover."* An automated pipeline with duplicate detection would have caught this.

---

## ✅ Mini Checkpoint

**Take 2 minutes to think:** Can you identify a data pipeline (manual or automated) that exists at your workplace or in a business you know?

Think about:
- What's the **source** of the data?
- What **processing** happens to it?
- Where does it **end up**?
- How **often** does it run?
- What happens when it **fails**?

---

<a id='section3'></a>
# Section 3 — ETL Deep Dive
### *Extract, Transform, Load — the three pillars of every pipeline*

---

ETL stands for **Extract**, **Transform**, **Load**. It's the standard framework for thinking about data pipelines. Let's go through each step with Ghana fintech examples.

---

## 📥 EXTRACT — Getting the Raw Data

The first step is simply getting the data from wherever it lives. In Ghana fintech, data comes from many sources:

| Source Type | Example | Format |
|------------|---------|--------|
| **REST API** | MTN MoMo Developer API | JSON |
| **SFTP Drop** | GhIPSS settlement files | CSV, ISO 8583 |
| **USSD Logs** | Transaction records from *170# sessions | Text/JSON |
| **Webhook Events** | Hubtel payment confirmations | JSON POST |
| **Database Export** | Daily dump from core banking system | SQL, CSV |
| **File Upload** | Agent reports from regional offices | Excel, CSV |
| **Message Queue** | Real-time transaction events | JSON messages |

### What makes extraction tricky?

- **Network timeouts**: Ghana's network coverage means API calls sometimes fail halfway through — you get partial data
- **Different formats**: One telco gives you JSON, another gives you CSV with different column names
- **Rate limits**: APIs often limit how many records you can fetch at once
- **Authentication**: Each system has its own auth method (API keys, OAuth, SSH certificates)
- **Timing**: End-of-day settlement data often isn't available until 11pm — your pipeline needs to wait for it

> 🇬🇭 **Ghana-specific challenge**: During load-shedding (dumsor), systems go offline. Your pipeline needs to handle the case where a partner system's data is missing for the day and either retry or flag it for manual review.

---

## 🔄 TRANSFORM — Cleaning and Enriching the Data

Raw data is almost never ready to use. The Transform step is where you:
- Fix inconsistencies
- Remove bad records
- Enrich data with additional information
- Standardize formats

### Common transformations for MoMo transaction data:

#### 1. Phone Number Normalization
The same phone number can appear in many formats:

```python
# Same number, different formats — all from the same day's data!
"0244123456"      # Local format (most common)
"+233244123456"  # International with + sign
"233244123456"   # International without + sign  
"+233 244 123 456"  # Spaced format
"0244-123-456"   # Dashed format

# After normalization, all become:
"+233244123456"
```

#### 2. Duplicate Transaction Detection
USSD sessions sometimes submit the same transaction twice when a customer loses network mid-session and retries. The **idempotency key** (usually `txn_id`) tells us if we've seen this transaction before.

```python
# These are duplicates — same txn_id, should only be processed once
{"txn_id": "TXN-ABC123", "amount": 500.00, "status": "PENDING"}
{"txn_id": "TXN-ABC123", "amount": 500.00, "status": "SUCCESS"}  # duplicate!
```

#### 3. Region Name Standardization
Data from different agent apps often uses different region names:

| Raw Value | Standardized Value |
|-----------|-------------------|
| `Gt. Accra` | `Greater Accra` |
| `greater accra` | `Greater Accra` |
| `Ashanti Region` | `Ashanti` |
| `Kumasi Region` | `Ashanti` |
| `Brong-Ahafo` | `Bono` |

#### 4. Currency Formatting
Always store monetary amounts as numbers, not strings. Always round to 2 decimal places for GHS.

```python
# Bad (string, inconsistent precision)
amount = "GHS 1,500.7"

# Good (float, 2 decimal places)
amount_ghs = 1500.70
```

#### 5. Status Validation
Valid statuses in our system: `SUCCESS`, `FAILED`, `PENDING`, `REVERSED`
Any other value (including `None`) needs to be handled:

```python
valid_statuses = {"SUCCESS", "FAILED", "PENDING", "REVERSED"}
if transaction["status"] not in valid_statuses:
    transaction["status"] = "UNKNOWN"  # flag for manual review
```

### Before and After: A Real Transformation Example

**Raw transaction (as received from agent app):**
```json
{
  "txn_id": "TXN-608B47E45D87",
  "timestamp": "2026-04-20T04:05:13Z",
  "sender_phone": "+233 244 123 456",
  "receiver_phone": "0501234567",
  "amount_ghs": 3383.819,
  "type": "CASH_IN",
  "status": null,
  "region": "Ashanti Region",
  "agent_id": "AGT-KSI-095",
  "network": "AirtelTigo"
}
```

**After transformation:**
```json
{
  "txn_id": "TXN-608B47E45D87",
  "timestamp": "2026-04-20T04:05:13Z",
  "sender_phone": "+233244123456",
  "receiver_phone": "+233501234567",
  "amount_ghs": 3383.82,
  "type": "CASH_IN",
  "status": "UNKNOWN",
  "region": "Ashanti",
  "agent_id": "AGT-KSI-095",
  "network": "AirtelTigo",
  "agent_commission_ghs": 16.92,
  "is_duplicate": false,
  "processed_at": "2026-04-20T22:01:45Z"
}
```

Notice what changed: phone numbers normalized, amount rounded, status set to UNKNOWN (was null), region standardized, commission calculated, metadata added.

---

## 📤 LOAD — Storing the Clean Data

After transformation, the data goes to its final destination. In our pipeline:

| Destination | Purpose | Example |
|-------------|---------|--------|
| **Amazon S3** (processed) | Archive clean data for audits and re-processing | `s3://fintech-processed/2026/04/20/` |
| **Amazon DynamoDB** | Fast transaction lookups by ID or date | Customer service queries |
| **Amazon Redshift** (advanced) | Analytics and reporting at scale | Monthly BoG reports |
| **Amazon RDS** (advanced) | Relational queries and joins | Multi-table reconciliation |

In this workshop, we'll use **S3** (processed zone) and **DynamoDB**.

---

## ⚡ Batch vs Streaming

There are two fundamental approaches to running pipelines:

### Batch Processing (What we're building)
- Process a **chunk of data at a scheduled time**
- Example: "Every night at 10pm, process all today's transactions"
- Most common in Ghanaian fintech — especially for end-of-day settlement
- Simpler to build and debug
- AWS Tools: Lambda + EventBridge, AWS Glue, EMR

### Streaming Processing (Advanced — for awareness)
- Process **each transaction as it arrives**, in real-time
- Example: "Flag this transaction as potentially fraudulent within 200ms of it being submitted"
- Used for real-time fraud detection, live dashboards
- More complex but more powerful
- AWS Tools: Amazon Kinesis, Apache Kafka (on AWS MSK)

> 💡 **Tip**: Start with batch. It's simpler, cheaper, and solves 90% of Ghanaian fintech data needs. Once you've mastered batch, learn streaming.

---

## 🧩 Data Quality Challenges — The Ghana Fintech Edition

Here are real data quality issues you'll encounter:

**1. Network Timeouts Causing Duplicates**
A customer dials `*170#`, initiates a GHS 200 transfer, but their network drops after submission. The USSD app retries automatically. You get two records for one transaction.

**2. Inconsistent Data from Multiple Telco Partners**
MTN's API returns `{"amount": 200.00, "currency": "GHS"}`. AirtelTigo's CSV export has `Amount (Cedis),200`. Vodafone's SFTP file has `TXN_AMT=20000` (in pesewas!). Your pipeline must handle all three.

**3. Missing Fields in USSD Transactions**
USSD has limited data capacity. A cash-out via `*170#` might not capture the receiver's name — only their phone number. Your pipeline needs to handle optional fields gracefully.

**4. Timestamp Timezone Confusion**
Ghana is on GMT+0 (no daylight saving). But some systems log in UTC, some in WAT (West Africa Time, UTC+1), some don't include timezone info at all. Always normalize to UTC.

**5. Agent Float Discrepancies**
An agent does a cash-in of GHS 500, but their float balance only shows GHS 400 deducted. Why? The GHS 100 is the agent's commission. If your pipeline doesn't account for this, your numbers won't balance.

---

## ✅ Mini Checkpoint

**Think about it:** Can you list **3 things that could go wrong** with raw MoMo transaction data before it's processed?

*(Hint: think about duplicates, formats, missing fields, network issues, human error)*

---

<a id='section4'></a>
# Section 4 — AWS Services for Fintech Data Pipelines
### *The tools we'll use — explained in plain language*

---

AWS (Amazon Web Services) is a cloud computing platform. Instead of buying and managing your own servers, you **rent computing resources on-demand** and pay only for what you use. For a startup fintech in Accra, this is game-changing — you don't need a server room, an IT team, or upfront hardware costs.

Let's meet the six AWS services we'll use, explained through analogies you'll recognize.

---

## 🗄️ Amazon S3 — Your Digital Warehouse

> *"Think of S3 like a large warehouse in Tema port. You can store anything there — raw goods in one section, processed goods in another. Each item has a label (key) so you can find it later. The warehouse is open 24/7 and never gets full."*

**What it is:** Amazon Simple Storage Service — an object storage service for storing any type of file at any scale.

**Key concepts:**
- **Bucket**: A named container for your files (like a folder at the top level)
  - Example: `fintech-pipeline-raw`, `fintech-pipeline-processed`
- **Object**: Any file stored in S3 — JSON, CSV, Parquet, images, anything
- **Key (prefix)**: The "path" to the object, like a folder structure
  - Example: `transactions/2026/04/20/batch_001.json`

**In our pipeline:**

```
s3://fintech-pipeline-raw/
    └── transactions/
        └── 2026/
            └── 04/
                └── 20/
                    └── batch_001.json   ← Raw, messy data lands here

s3://fintech-pipeline-processed/
    └── transactions/
        └── 2026/
            └── 04/
                └── 20/
                    └── batch_001.parquet  ← Clean data goes here
```

**Why S3?**
- 99.999999999% durability — your data won't disappear
- Costs as little as $0.023 per GB per month (< GHS 0.35)
- Unlimited storage — grow from 500 transactions to 500 million without changing anything
- Can trigger Lambda automatically when new files arrive
- **Free Tier**: 5GB free for 12 months

---

## ⚡ AWS Lambda — The Worker Who Sleeps Until Called

> *"Lambda is like a casual worker (daily laborer) who sleeps at home. When you call them, they come, do exactly the job you describe, get paid, and go back home. You only pay for the exact hours they work — not the hours they're asleep."*

**What it is:** A serverless compute service. You write a Python function, upload it to Lambda, and AWS runs it when triggered. You don't manage any servers.

**Key concepts:**
- **Function**: Your Python code (max 15 minutes runtime per invocation)
- **Trigger**: What causes Lambda to run (S3 event, schedule, API call)
- **Handler**: The entry point function AWS calls (`def lambda_handler(event, context):`)
- **Serverless**: No servers to manage — AWS handles scaling automatically

**What our Lambda does:**
```python
def lambda_handler(event, context):
    # 1. Called when new file lands in S3 raw bucket
    # 2. Reads the raw transactions
    # 3. Cleans and transforms them  
    # 4. Saves clean data to S3 processed bucket
    # 5. Loads into DynamoDB
    # 6. Sends alert if failures detected
    pass
```

**Why Lambda?**
- **Free Tier**: 1,000,000 requests per month FREE — you'll never exceed this at small scale
- Only pay when running: processing 500 transactions takes ~2 seconds → costs almost nothing
- Scales automatically: 1 file or 10,000 files — same Lambda handles it
- Native Python support

---

## ⏰ Amazon EventBridge — The Alarm Clock for Your Pipeline

> *"EventBridge is like setting an alarm on your phone: every night at 10pm, it rings and wakes up your Lambda function to start processing the day's transactions."*

**What it is:** A serverless event bus and scheduler. You define **rules** that trigger actions on a schedule or in response to events.

**Cron expressions** — the language of schedules:

```
cron(Minutes Hours Day Month DayOfWeek Year)

cron(0 22 * * ? *)   → Every day at 10:00 PM UTC (midnight Ghana time)
cron(0 6 * * ? *)    → Every day at 6:00 AM UTC (morning Ghana time)
cron(0 22 ? * MON-FRI *)  → Weekdays only at 10PM UTC
cron(0 */4 * * ? *) → Every 4 hours
```

> 🇬🇭 **Ghana time note**: Ghana uses GMT (UTC+0) year-round — no daylight saving. So 10pm UTC = 10pm Ghana time. Simple!

---

## 📒 Amazon DynamoDB — The Super-Fast Transaction Ledger

> *"DynamoDB is like a very organized clerk who can find any transaction record in milliseconds. You give them the transaction ID, they hand you the full record instantly. No searching through piles of paper."*

**What it is:** A fully managed NoSQL database. Unlike SQL databases (rows and columns), DynamoDB stores **items** (like JSON objects) and retrieves them by key.

**Key concepts:**
- **Table**: Collection of items (like a database table)
- **Item**: A single record (like a row) — stored as key-value pairs
- **Partition Key**: The primary identifier — must be unique (e.g., `txn_id`)
- **Sort Key**: Secondary identifier for ordering within a partition (e.g., `timestamp`)

**Example item in DynamoDB:**
```json
{
  "txn_id": "TXN-608B47E45D87",     ← Partition Key
  "timestamp": "2026-04-20T04:05:13Z", ← Sort Key
  "amount_ghs": 3383.82,
  "type": "CASH_IN",
  "status": "SUCCESS",
  "region": "Ashanti",
  "agent_id": "AGT-KSI-095"
}
```

**Why DynamoDB for transaction lookups?**
- Retrieves any record by `txn_id` in under **10 milliseconds** — critical for customer service
- Handles millions of concurrent reads/writes without slowing down
- **Free Tier**: 25GB storage + 25 read/write capacity units FREE permanently
- Perfect for "give me transaction TXN-XYZ" queries

**When NOT to use DynamoDB:**
- Complex SQL queries with multiple JOINs → use RDS or Redshift instead
- Large analytical aggregations across millions of rows → use Redshift or Athena

---

## 📢 Amazon SNS — The Megaphone That Shouts When Things Go Wrong

> *"SNS is like a town crier. When your pipeline detects 50 failed transactions, SNS shouts the news to everyone who subscribed — the team lead gets an email, the on-call engineer gets an SMS, the Slack channel gets a notification."*

**What it is:** Simple Notification Service — a publish/subscribe messaging service for sending alerts.

**Key concepts:**
- **Topic**: A named channel for a category of alerts (e.g., `pipeline-alerts`)
- **Subscription**: Who gets notified (email, SMS, Lambda, SQS queue)
- **Message**: The alert content you publish

**In our pipeline:**
```
Lambda detects 23 FAILED transactions
    → Publishes to SNS topic: "pipeline-alerts"
        → Email to: fintech-team@company.com.gh
        → SMS to: +233244XXXXXX (team lead's phone)
```

**Why this matters for compliance:**
Bank of Ghana expects fintech companies to have incident response procedures. An automated SNS alert is your first line of defence — the team knows about failures within seconds, not hours.

**Free Tier**: 1,000,000 SNS publishes FREE per month

---

## 🔍 Amazon CloudWatch — CCTV for Your Pipeline

> *"CloudWatch is like the CCTV system in a bank. It records everything that happens, raises an alarm when something suspicious occurs, and lets you replay the footage (logs) to investigate incidents."*

**What it is:** AWS's monitoring and observability service. It collects logs, metrics, and events from all your AWS services automatically.

**Key capabilities:**

| Feature | What it does | Fintech example |
|---------|-------------|----------------|
| **Logs** | Store all print/log statements from Lambda | "Lambda processed 500 txns at 22:01" |
| **Metrics** | Track numbers over time | Invocation count, error rate, duration |
| **Alarms** | Alert when a metric crosses a threshold | "Error rate > 5% → send SNS alert" |
| **Dashboards** | Visual display of your metrics | Real-time pipeline health view |

**What a CloudWatch dashboard might show:**
```
┌──────────────────────────────────────────────────────────┐
│           Pipeline Health Dashboard — 2026-04-20         │
├─────────────────┬──────────────┬───────────┬────────────┤
│  Invocations    │  Error Rate  │  Duration │  Records   │
│      1          │    0.0%      │  2.3 sec  │    488     │
│  ✅ Healthy     │  ✅ Normal   │ ✅ Fast   │ ✅ Loaded  │
└─────────────────┴──────────────┴───────────┴────────────┘
```

**Free Tier**: 10 custom metrics, 3 dashboards, 5GB log data FREE per month

---

## 🔒 Security Essentials (Brief Overview)

Security is non-negotiable in fintech. Here are the essentials:

### IAM — Identity and Access Management
**Principle of Least Privilege**: Give each service only the exact permissions it needs.

```
Lambda's IAM Role permissions:
✅ s3:GetObject (read from raw bucket)
✅ s3:PutObject (write to processed bucket)
✅ dynamodb:PutItem (write transactions)
✅ sns:Publish (send alerts)
❌ s3:DeleteBucket (definitely NOT needed)
❌ iam:CreateUser (definitely NOT needed)
```

### Encryption
- **S3**: Server-Side Encryption (SSE-S3) enabled by default — your data is encrypted at rest
- **DynamoDB**: Encryption at rest enabled by default
- **In transit**: All AWS API calls use HTTPS/TLS

### Why This Matters for Ghana
The **Ghana Data Protection Act (Act 843, 2012)** requires organizations processing personal data to implement appropriate security measures. Mobile money transaction data includes personal information (names, phone numbers, financial data) — it falls squarely under this Act.

The **Bank of Ghana's EMI Guidelines** also require that transaction records be retained for a minimum of **7 years** with appropriate access controls.

---

## 🏗️ Full Architecture — Putting It All Together

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    COMPLETE PIPELINE ARCHITECTURE                       │
│                                                                         │
│  ① DATA ARRIVES          ② STORAGE             ③ PROCESSING            │
│  ─────────────          ───────────            ────────────            │
│  MoMo Agent API  ──────▶ S3 Raw Bucket ──────▶ AWS Lambda             │
│  (JSON batch)             fintech-raw/           (Python code)         │
│                           txns/2026/04/20/        • Read from S3       │
│                                                   • Clean data         │
│                                                   • Deduplicate        │
│  ④ CLEAN STORAGE         ⑤ FAST LOOKUP           • Normalize          │
│  ────────────────        ─────────────           └──────────┬──────────┘
│  S3 Processed   ◀──────  DynamoDB Table ◀──────────────────┘          │
│  fintech-proc/            momo_transactions                             │
│  txns/2026/04/20/         PK: txn_id                                   │
│                           SK: timestamp                                 │
│                                                                         │
│  ⑥ ALERTING              ⑦ MONITORING           ⑧ SCHEDULING          │
│  ───────────             ────────────           ────────────           │
│  Amazon SNS      ◀───── CloudWatch    ◀──────── EventBridge            │
│  Email/SMS alerts         Logs, Metrics          cron(0 22 * * ? *)    │
│  to fintech team          Alarms                 Runs at 10pm UTC      │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## ✅ Mini Checkpoint — Match the Service

Can you match each AWS service to its role?

| Service | Role |
|---------|------|
| Amazon S3 | A. Sends email/SMS alerts when pipeline fails |
| AWS Lambda | B. Stores raw and processed transaction files |
| Amazon EventBridge | C. Runs the pipeline code serverlessly |
| Amazon DynamoDB | D. Lets you look up any transaction in milliseconds |
| Amazon SNS | E. Schedules the pipeline to run every night |
| Amazon CloudWatch | F. Monitors pipeline health and stores logs |

*(Answers: S3→B, Lambda→C, EventBridge→E, DynamoDB→D, SNS→A, CloudWatch→F)*

---

<a id='section5'></a>
# Section 5 — Best Practices & Compliance
### *Building pipelines that are production-ready and regulation-compliant*

---

## 🔁 Idempotency — The Most Important Word in Fintech Engineering

> *"If your Lambda runs twice, it should NOT debit the customer twice."*

**Idempotency** means: running an operation multiple times produces the same result as running it once.

This is critical in fintech because:
- Networks fail → your Lambda might get triggered twice for the same file
- EventBridge might fire twice due to a glitch
- A retry mechanism might re-process already-processed data

**How to implement idempotency:**

```python
def load_transaction(txn):
    try:
        # condition_expression ensures we don't overwrite if txn_id already exists
        table.put_item(
            Item=txn,
            ConditionExpression="attribute_not_exists(txn_id)"
        )
    except dynamodb.meta.client.exceptions.ConditionalCheckFailedException:
        # This transaction was already loaded — that's OK, just log it
        logger.info(f"Transaction {txn['txn_id']} already exists — skipping")
```

---

## 📋 Logging for Audit Trails

The Bank of Ghana requires that every transaction be **traceable**. Your pipeline logs are part of that audit trail. Log:

- **When** each batch was processed
- **How many** transactions were received, cleaned, and loaded
- **Which** transactions were flagged as duplicates or invalid
- **Any errors** that occurred and what happened next

```python
import logging
logger = logging.getLogger(__name__)

logger.info("Pipeline started — batch: batch_001.json")
logger.info("Extracted 500 transactions from S3")
logger.warning("12 duplicate transactions flagged and removed")
logger.info("488 clean transactions loaded into DynamoDB")
logger.info("Pipeline completed successfully in 2.3 seconds")
```

CloudWatch automatically captures these log entries and stores them for **7 years** if configured (matching BoG retention requirements).

---

## 🔄 Error Handling Strategies

### Retry with Backoff
If an API call fails, wait before retrying — the service might just be temporarily overloaded:

```python
import time

def call_api_with_retry(url, max_retries=3):
    for attempt in range(max_retries):
        try:
            return requests.get(url)
        except Exception as e:
            wait_seconds = 2 ** attempt  # 1s, 2s, 4s
            logger.warning(f"Attempt {attempt+1} failed. Retrying in {wait_seconds}s")
            time.sleep(wait_seconds)
    raise Exception("All retries failed")
```

### Dead Letter Queues (Concept)
When a Lambda fails repeatedly, AWS can send the failed event to a **Dead Letter Queue (DLQ)** — an SQS queue — instead of losing it. Your team can then investigate and re-process those failed events later.

---

## 🗂️ Data Retention Policies

| Data Type | Retention Period | Reason |
|-----------|-----------------|--------|
| Raw transaction files (S3) | 7 years | Bank of Ghana compliance |
| Processed/clean data (S3) | 7 years | Audit trail |
| DynamoDB records | 2 years (then archive to S3) | Cost optimization |
| CloudWatch logs | 1 year | Investigation needs |
| SNS alert history | 30 days | Operational review |

Configure S3 Lifecycle rules to automatically move old data to cheaper storage (S3 Glacier) after 1 year.

---

## ⚖️ Ghana Data Protection Act (Act 843, 2012)

The Ghana Data Protection Act applies to any organization that processes **personal data** — which includes mobile money transaction data because it contains:
- Full names
- Phone numbers
- Financial transaction amounts
- Location data (region)

**Key obligations for your pipeline:**

| Obligation | What It Means for Your Pipeline |
|-----------|--------------------------------|
| **Purpose limitation** | Only collect data needed for the stated purpose |
| **Data security** | Encrypt data at rest and in transit (S3 + DynamoDB do this automatically) |
| **Access control** | Only authorized personnel can access transaction data (use IAM roles) |
| **Data retention** | Don't keep data longer than necessary (use lifecycle policies) |
| **Breach notification** | If data is compromised, notify the Data Protection Commission |

> ⚠️ **Important**: If you're processing personal data of Ghanaian citizens at scale, your organization should register with the **Data Protection Commission (DPC)** at `dataprotection.org.gh`.

---

## 💰 Cost Awareness — Staying in the Free Tier

Everything we build today fits comfortably within the **AWS Free Tier**:

| Service | Free Tier Limit | Our Usage |
|---------|----------------|----------|
| S3 | 5 GB storage, 20,000 GET requests | < 10 MB, < 100 requests |
| Lambda | 1,000,000 requests, 400,000 GB-seconds | < 100 requests |
| DynamoDB | 25 GB, 25 RCU/WCU | < 1 GB, < 1 RCU/WCU |
| SNS | 1,000,000 publishes, 1,000 SMS | < 10 publishes |
| CloudWatch | 10 custom metrics, 5 GB logs | < 5 metrics |

**Estimated cost for this workshop: $0.00** ✅

---

## 🎯 Section 5 Recap

Before you write pipeline code, always think about:

1. ✅ **Idempotency** — can this run twice safely?
2. ✅ **Logging** — will you be able to explain what happened 6 months later?
3. ✅ **Error handling** — what happens when things go wrong?
4. ✅ **Data retention** — how long do you keep data and in what tier?
5. ✅ **Compliance** — are you meeting BoG and DPC requirements?

---

# 🏁 Theory Complete! You're Ready to Build

Congratulations on working through the theory notebook! Let's recap what you now know:

| Topic | You Now Understand |
|-------|-------------------|
| Data pipelines | What they are, why they matter in Ghana fintech |
| ETL | Extract from sources, Transform dirty data, Load to databases |
| AWS Services | S3, Lambda, EventBridge, DynamoDB, SNS, CloudWatch |
| Best Practices | Idempotency, logging, error handling, compliance |
| Architecture | How all the pieces connect into a complete system |

## 👉 Next: Open `02_practical_labs.ipynb`

In the lab notebook, you'll:
- Generate 500 realistic MoMo transactions (with intentionally dirty data)
- Upload them to a mock S3 bucket
- Clean and transform the data with Python
- Load into a mock DynamoDB table
- Wire it all together into an automated pipeline
- Add monitoring and alerts

The labs use **moto** — a library that mocks AWS services locally. You don't need an AWS account. Everything runs right in this Jupyter environment.

---

> 🇬🇭 *"Fintech in Ghana is growing fast. The engineers who understand data pipelines will be the ones building the infrastructure that powers the next MTN MoMo, the next Zeepay, the next GhIPSS. That could be you."*

**Good luck, and let's build! 🚀**